In [ ]:
import finnhub
import os
import time
from dotenv import load_dotenv

load_dotenv()
finnhub_client = finnhub.Client(api_key=os.getenv("FINNHUB_API_KEY"))

FROM_DATE = "2023-01-01"
TO_DATE = "2025-12-31"

data = finnhub_client.stock_usa_spending(symbol="AAPL", _from=FROM_DATE, to=TO_DATE)

In [2]:
print("=== TOP LEVEL KEYS ===")
for key in data.keys():
    print(f"  {key}")

print(f"\nSpending activities returned: {len(data['data'])}")

=== TOP LEVEL KEYS ===
  data
  symbol

Spending activities returned: 7


In [3]:
print("=== FIRST ACTIVITY ===")
if data["data"]:
    for k, v in data["data"][0].items():
        print(f"  {k}: {v}")
else:
    print("  No spending activities found for AAPL in this date range")

=== FIRST ACTIVITY ===
  symbol: AAPL
  recipientName: APPLE INC
  recipientParentName: APPLE INC
  country: USA
  totalValue: 13585
  outlayedAmount: 0
  obligatedAmount: 13585
  potentialAmount: 13585
  actionDate: 2025-09-30
  performanceStartDate: 2025-09-30
  performanceEndDate: 2025-11-29
  awardingAgencyName: Department of State
  awardingSubAgencyName: Department of State
  awardingOfficeName: U.S. EMBASSY PARIS
  performanceCountry: FRA
  performanceCity: 
  performanceCounty: 
  performanceState: 
  performanceZipCode: 
  performanceCongressionalDistrict: CA-17
  awardDescription: TELECOM CONSULAR PROJECT NEW DESK PHONE PART
  naicsCode: 334210
  permalink: https://www.usaspending.gov/award/CONT_AWD_19FR6325K1263_1900_-NONE-_-NONE-/
  lastModifiedDate: 2025-10-09


In [4]:
print("=== SPENDING ACTIVITIES SUMMARY ===")
if data["data"]:
    for activity in data["data"]:
        print(f"  {activity['actionDate']} | {activity['awardingAgencyName']} | totalValue: {activity['totalValue']} | {activity['awardDescription'][:50] if activity['awardDescription'] else 'N/A'}")
else:
    print("  No activities to display")

=== SPENDING ACTIVITIES SUMMARY ===
  2025-09-30 | Department of State | totalValue: 13585 | TELECOM CONSULAR PROJECT NEW DESK PHONE PART
  2025-09-29 | Agency for International Development | totalValue: 0 | APPLE PRODUCTS AND SERVICES
  2025-09-22 | Department of State | totalValue: 13598.11 | TELEPHONE DEVICES
  2025-09-15 | Department of State | totalValue: 12283.64 | SMARTPHONE
  2025-06-11 | Department of State | totalValue: 63555.61 | ICASS/PROG: IPHONE 13 PRO/PRO MAX, UNLOCKED 128 GB
  2025-05-16 | Department of Justice | totalValue: 7470 | CREDIT CARD PURCHASE POP DATES: 09/29/2025 TO 09/2
  2025-01-01 | Department of Justice | totalValue: 299 | APPLE DEVELOPER ACCOUNT RENEWAL


### ─────────────────────────────────────────────
### SECTION 2 — PRODUCTION RUN (ALL 60 COMPANIES)
### ─────────────────────────────────────────────

In [ ]:
import finnhub
import os
from dotenv import load_dotenv

load_dotenv()
finnhub_client = finnhub.Client(api_key=os.getenv("FINNHUB_API_KEY"))

FROM_DATE = "2023-01-01"
TO_DATE = "2025-12-31"

tickers = [
    # DEFENSE - High Lobby
    "LMT", "RTX", "NOC", "GD", "BA", "LHX", "LDOS", "HII", "BAESY", "SAIC",
    # DEFENSE - Low Lobby
    "TXT", "TDG", "HEI", "DRS", "KTOS", "AVAV", "MRCY", "CW", "MOG.A", "DCO",
    # ENERGY - High Lobby
    "XOM", "CVX", "COP", "OXY", "BP", "NEE", "D", "DUK", "HAL", "BKR",
    # ENERGY - Low Lobby
    "SLB", "VLO", "PSX", "EOG", "FANG", "DVN", "CTRA", "AR", "CHRD", "MTDR",
    # TECH - High Lobby
    "MSFT", "AMZN", "GOOGL", "IBM", "ORCL", "PLTR", "BAH", "CACI", "PSN", "CRM",
    # TECH - Low Lobby
    "AAPL", "META", "NVDA", "CSCO", "PANW", "CRWD", "SNOW", "DDOG", "NET", "TWLO"
]

# Duplicate guard
duplicates = [t for t in tickers if tickers.count(t) > 1]
assert not duplicates, f"Duplicate tickers found: {duplicates}"

results = {}
issues = []

OPTIONAL_FIELDS = {
    "performanceCity", "performanceCongressionalDistrict", "performanceCounty",
    "performanceZipCode", "performanceState", "awardingOfficeName",
    "awardingSubAgencyName", "naicsCode", "recipientParentName"
}

for i, ticker in enumerate(tickers):
    try:
        data = finnhub_client.stock_usa_spending(symbol=ticker, _from=FROM_DATE, to=TO_DATE)
        results[ticker] = data

        # No spending activity is meaningful data, not an error
        if not data or not data.get("data"):
            issues.append((ticker, "no spending activities found in date range"))
            continue

        # Check for None fields in each activity, ignoring optional fields
        for activity in data["data"]:
            none_fields = [
                k for k, v in activity.items()
                if (v is None or v == "") and k not in OPTIONAL_FIELDS
            ]
            if none_fields:
                issues.append((ticker, f"missing fields in activity {activity.get('actionDate')}: {none_fields}"))
            break  # flag once per company

    except Exception as e:
        issues.append((ticker, f"API error: {str(e)}"))
        results[ticker] = {}

    if i < len(tickers) - 1:
        time.sleep(2)  # 2 seconds — consistent with lobbying endpoint

# Summary
successful = [t for t, r in results.items() if r and r.get("data")]
no_activity = [t for t, r in results.items() if r is not None and not r.get("data")]

print(f"✅ Successfully pulled: {len(results)} / {len(tickers)} companies")
print(f"📋 Companies with spending activity: {len(successful)}")
print(f"⭕ Companies with no spending activity: {len(no_activity)}")
print(f"   {no_activity}")
print(f"⚠️  Issues found: {len(issues)}")
for ticker, issue in issues:
    print(f"   {ticker}: {issue}")

✅ Successfully pulled: 60 / 60 companies
📋 Companies with spending activity: 45
⭕ Companies with no spending activity: 15
   ['DRS', 'COP', 'HAL', 'BKR', 'EOG', 'FANG', 'DVN', 'CTRA', 'AR', 'CHRD', 'MTDR', 'NVDA', 'PANW', 'CRWD', 'SNOW']
⚠️  Issues found: 21
   BA: missing fields in activity 2025-12-30: ['performanceEndDate', 'performanceCountry']
   LHX: missing fields in activity 2025-12-31: ['performanceEndDate', 'performanceCountry']
   BAESY: missing fields in activity 2025-12-17: ['performanceEndDate', 'performanceCountry']
   TXT: missing fields in activity 2025-12-29: ['performanceEndDate', 'performanceCountry']
   DRS: no spending activities found in date range
   COP: no spending activities found in date range
   HAL: no spending activities found in date range
   BKR: no spending activities found in date range
   SLB: missing fields in activity 2025-05-27: ['performanceEndDate', 'performanceCountry']
   EOG: no spending activities found in date range
   FANG: no spending acti